In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import os
from pathlib import Path
from sklearn.model_selection import train_test_split

In [ ]:
# =============================================================================
# DATASET CLASS
# =============================================================================

class MultiLabelImageDataset(Dataset):
    """
    Custom Dataset for multi-label image classification.
    
    Expected CSV format:
    - Column 0: image filename/path
    - Columns 1-15: binary labels (0 or 1) for each class
    
    Or with a 'labels' column containing comma-separated label indices.
    """
    def __init__(self, 
                 image_paths, 
                 labels, 
                 transform=None, 
                 img_dir=None):
        """
        Args:
            image_paths: List or array of image paths/filenames
            labels: Array of shape (N, num_classes) with binary labels
            transform: torchvision transforms to apply
            img_dir: Base directory containing images (optional)
        """
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.img_dir = Path(img_dir) if img_dir else None
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image
        img_path = self.image_paths[idx]
        if self.img_dir:
            img_path = self.img_dir / img_path
        
        image = Image.open(img_path).convert('RGB')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        # Get labels (convert to float tensor)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        return image, label

In [ ]:
# =============================================================================
# DATA LOADING FUNCTIONS
# =============================================================================

def load_and_split_data(csv_path, img_dir, class_names, test_size=0.2, random_state=42):
    """
    Load dataset from CSV file and perform stratified split.
    Matches the TensorFlow preprocessing workflow.
    
    Args:
        csv_path: Path to CSV file
        img_dir: Directory containing images
        class_names: List of class/label column names
        test_size: Proportion for validation set
        random_state: Random seed for reproducibility
    
    Returns:
        train_df, val_df, class_counts
    """
    # Load CSV
    df = pd.read_csv(csv_path)
    
    # Keep only Path and class columns
    df = df[['Path'] + class_names]
    
    print(f"Dataset loaded: {len(df)} samples")
    print(f"Columns: {df.columns.tolist()}")
    
    # Calculate class distribution
    class_counts = df[class_names].sum()
    print("\nClass Distribution:")
    print("=" * 50)
    for label, count in zip(class_names, class_counts):
        percentage = (count / len(df)) * 100
        print(f"{label:20s}: {count:5d} ({percentage:5.2f}%)")
    
    # Create stratification column (has any disease or not)
    df['has_any_disease'] = (df[class_names].sum(axis=1) > 0).astype(int)
    
    # Stratified split
    train_df, val_df = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        stratify=df['has_any_disease']
    )
    
    print(f"\nDataset Split:")
    print(f"Train: {len(train_df)} samples")
    print(f"Validation: {len(val_df)} samples")
    
    return train_df, val_df, class_counts


def create_data_loaders(
    train_df,
    val_df,
    class_names,
    img_dir,
    batch_size=32,
    img_size=224,
    num_workers=4
):
    """
    Create train and validation data loaders from DataFrames.
    
    Args:
        train_df: Training DataFrame with 'Path' and class columns
        val_df: Validation DataFrame with 'Path' and class columns
        class_names: List of class/label column names
        img_dir: Directory containing images
        batch_size: Batch size for training
        img_size: Image size for resizing
        num_workers: Number of workers for data loading
    
    Returns:
        train_loader, val_loader, pos_weights
    """
    
    # Extract paths and labels
    train_paths = train_df['Path'].values
    train_labels = train_df[class_names].values.astype(np.float32)
    
    val_paths = val_df['Path'].values
    val_labels = val_df[class_names].values.astype(np.float32)
    
    # Calculate class weights for WBCE (based on training set)
    pos_counts = train_labels.sum(axis=0)
    neg_counts = len(train_labels) - pos_counts
    
    # Avoid division by zero
    pos_counts = np.maximum(pos_counts, 1)
    neg_counts = np.maximum(neg_counts, 1)
    
    # Weight = neg_samples / pos_samples (higher weight for rare classes)
    pos_weights = neg_counts / pos_counts
    
    print(f"\nClass weights for WBCE (based on training set):")
    print("=" * 50)
    for label, weight in zip(class_names, pos_weights):
        print(f"{label:20s}: {weight:.4f}")
    
    # Define transforms
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    # Create datasets
    train_dataset = MultiLabelImageDataset(
        train_paths, train_labels, transform=train_transform, img_dir=img_dir
    )
    val_dataset = MultiLabelImageDataset(
        val_paths, val_labels, transform=val_transform, img_dir=img_dir
    )
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,
        pin_memory=True
    )
    
    return train_loader, val_loader, pos_weights

In [ ]:
# =============================================================================
# WEIGHTED BCE LOSS
# =============================================================================

class WeightedBCEWithLogitsLoss(nn.Module):
    """
    Weighted Binary Cross Entropy Loss with Logits.
    
    This combines sigmoid activation and BCE loss with class weights.
    The pos_weight parameter increases the recall for positive classes,
    which is useful for imbalanced datasets.
    
    Formula per class:
    loss = -[w_pos * y * log(σ(x)) + (1-y) * log(1-σ(x))]
    where w_pos is the positive class weight
    """
    def __init__(self, pos_weights):
        """
        Args:
            pos_weights: Tensor of shape (num_classes,) with positive class weights
        """
        super().__init__()
        self.pos_weights = pos_weights
        
    def forward(self, outputs, targets):
        return nn.functional.binary_cross_entropy_with_logits(
            outputs, 
            targets, 
            pos_weight=self.pos_weights
        )


In [ ]:
# =============================================================================
# TRAINING FUNCTION
# =============================================================================

def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
    
    epoch_loss = running_loss / len(train_loader.dataset)
    return epoch_loss


def validate(model, val_loader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            
            # Get predictions (apply sigmoid and threshold at 0.5)
            preds = torch.sigmoid(outputs) > 0.5
            
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(val_loader.dataset)
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    # Calculate accuracy (exact match ratio)
    accuracy = (all_preds == all_labels).all(axis=1).mean()
    
    return epoch_loss, accuracy

In [ ]:
# =============================================================================
# COMPLETE TRAINING SCRIPT
# =============================================================================

def train_model(
    model,
    train_loader,
    val_loader,
    pos_weights,
    device,
    num_epochs=50,
    freeze_epochs=10,
    lr_initial=1e-3,
    lr_finetune=1e-4
):
    """
    Complete training script with frozen and unfrozen phases.
    
    Args:
        model: The ResNetViTHybrid model
        train_loader: Training data loader
        val_loader: Validation data loader
        pos_weights: Positive class weights for WBCE
        device: torch device
        num_epochs: Total number of epochs
        freeze_epochs: Number of epochs to keep CNN frozen
        lr_initial: Learning rate for frozen phase
        lr_finetune: Learning rate for fine-tuning phase
    """
    
    # Move model to device
    model = model.to(device)
    
    # Convert pos_weights to tensor and move to device
    pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)
    
    # Define loss function
    criterion = WeightedBCEWithLogitsLoss(pos_weights_tensor)
    
    # Phase 1: Train with frozen CNN backbone
    print("=" * 70)
    print("PHASE 1: Training with frozen ResNet backbone")
    print("=" * 70)
    
    model.freeze_cnn_backbone()
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr_initial
    )
    
    best_val_loss = float('inf')
    
    for epoch in range(freeze_epochs):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        print(f"Epoch {epoch+1}/{freeze_epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f}")
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model_frozen.pth')
    
    # Phase 2: Fine-tune entire model
    print("\n" + "=" * 70)
    print("PHASE 2: Fine-tuning entire model")
    print("=" * 70)
    
    model.unfreeze_cnn_backbone()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr_finetune)
    
    # Optionally use a learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )
    
    for epoch in range(freeze_epochs, num_epochs):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f}")
        
        # Update learning rate
        scheduler.step(val_loss)
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model_finetuned.pth')
    
    print("\nTraining complete!")
    return model

In [ ]:
# =============================================================================
# EXAMPLE USAGE
# =============================================================================

if __name__ == "__main__":
    # Import the model from your file
    from your_model_file import ResNetViTHybrid, CONFIG
    
    # Configuration matching TensorFlow workflow
    BASE_DIR = "./dataset_balanced/"
    CSV_PATH = os.path.join(BASE_DIR, "new_labels.csv")
    IMAGE_DIR = os.path.join(BASE_DIR, "new_images")
    
    # Define class names (same order as in your CSV)
    CLASS_NAMES = [
        'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
        'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
        'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
        'Pleural_Thickening', 'Hernia', 'No Finding'
    ]
    
    # Verify paths exist
    if not os.path.exists(CSV_PATH):
        print(f"Warning: CSV file not found at {CSV_PATH}")
    if not os.path.exists(IMAGE_DIR):
        print(f"Warning: Image directory not found at {IMAGE_DIR}")
    
    BATCH_SIZE = 32
    NUM_EPOCHS = 50
    FREEZE_EPOCHS = 10
    
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")
    
    # Step 1: Load and split data (matching TensorFlow workflow)
    print("Loading and splitting dataset...")
    train_df, val_df, class_counts = load_and_split_data(
        csv_path=CSV_PATH,
        img_dir=IMAGE_DIR,
        class_names=CLASS_NAMES,
        test_size=0.2,
        random_state=42
    )
    
    # Step 2: Create data loaders
    print("\nCreating data loaders...")
    train_loader, val_loader, pos_weights = create_data_loaders(
        train_df=train_df,
        val_df=val_df,
        class_names=CLASS_NAMES,
        img_dir=IMAGE_DIR,
        batch_size=BATCH_SIZE,
        img_size=CONFIG["IMAGE_SIZE"],
        num_workers=4
    )
    
    # Create model
    print("\nCreating model...")
    model = ResNetViTHybrid(config=CONFIG)
    
    # Train model
    print("\nStarting training...")
    model = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        pos_weights=pos_weights,
        device=device,
        num_epochs=NUM_EPOCHS,
        freeze_epochs=FREEZE_EPOCHS,
        lr_initial=1e-3,
        lr_finetune=1e-4
    )
    
    # Final evaluation
    print("\nFinal evaluation on validation set...")
    model.load_state_dict(torch.load('best_model_finetuned.pth'))
    pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)
    criterion = WeightedBCEWithLogitsLoss(pos_weights_tensor)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    print(f"Final Validation Loss: {val_loss:.4f} | Accuracy: {val_acc:.4f}")